In [1]:
import hashlib
import json
import os
import re
from pathlib import Path
import uuid

import boto3
from botocore.config import Config
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http import models

from mistralai.client import Mistral

In [2]:
qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL", "http://213.136.80.53:6333"))

In [3]:
qdrant_client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='legal_acts_event_rag'), CollectionDescription(name='legal_acts_event_rag_full')])

In [4]:
load_dotenv(Path("../.env"))
load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL", "http://213.136.80.53:6333")
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION", "legal_acts_event_rag_full")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
ACTS_DIR = Path("../data/acts")

MAX_CHARS_PER_CHUNK = 1200
CHUNK_OVERLAP = 100
UPSERT_BATCH_SIZE = 32

if not EMBEDDING_MODEL:
    raise ValueError("EMBEDDING_MODEL is required in .env")

json_files = sorted(ACTS_DIR.glob("act-print-*.json"))
len(json_files)

1484

In [5]:
EMBEDDING_MODEL

'cohere.embed-multilingual-v3'

In [6]:
SECTION_INDEX_RE = re.compile(
    r"^[\s\"'\[\]]*\[?([0-9\u09E6-\u09EF]+[a-zA-Z]*)[.\u0964\u09F7\-\s]"
)
FOOTNOTE_MARKER_RE = re.compile(r"\d+\[(.*?)\]")
# SECTION_REPEAL_RE = re.compile(r"\[\s*Repealed\s+by|\[\s*Repeal\.\-", re.IGNORECASE)
VOID_SECTION_RE = re.compile(
    r"\[\s*(Omitted|Repealed?|Rep\.)\s+by" r"|\[\s*Repeal\.\-" r"|\[\s*Omit\.\-",
    re.IGNORECASE,
)
SUBSECTION_SPLIT_RE = re.compile(r"(?=(?:\(\d+\)|\([a-zA-Z]+\)))")

In [ ]:
def extract_section_index(section_content: str) -> str:
    if not section_content:
        return "Unknown"
    match = SECTION_INDEX_RE.search(section_content)
    return (match.group(1).strip() if match else "Unknown") or "Unknown"

In [ ]:
def clean_section_content(section_content: str) -> str:
    if not section_content:
        return ""
    return FOOTNOTE_MARKER_RE.sub(r"\1", section_content).strip()

In [ ]:
def chunk_section_content(
    text: str, max_chars: int = 1200, overlap: int = 100
) -> list[str]:
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    def slice_with_overlap(value: str) -> list[str]:
        chunks, start = [], 0
        while start < len(value):
            end = min(start + max_chars, len(value))
            chunks.append(value[start:end].strip())
            if end >= len(value):
                break
            start = max(0, end - overlap)
        return [chunk for chunk in chunks if chunk]

    # Prefer semantic-ish splits on subsection markers before fixed-size slicing.
    parts = [part.strip() for part in SUBSECTION_SPLIT_RE.split(text) if part.strip()]
    if len(parts) > 1:
        merged, current = [], ""
        for part in parts:
            candidate = f"{current} {part}".strip() if current else part
            if len(candidate) <= max_chars:
                current = candidate
                continue

            if current:
                merged.append(current)
                current = ""

            if len(part) <= max_chars:
                current = part
                continue

            merged.extend(slice_with_overlap(part))

        if current:
            merged.append(current)
        if merged:
            return merged

    return slice_with_overlap(text)

In [ ]:
def build_embedding_text(
    act_title: str, section_index: str, chunk_part: int, chunk_text: str
) -> str:
    return (
        f"Act: {act_title}\nSection {section_index} (Part {chunk_part}): {chunk_text}"
    )

In [ ]:
def collect_all_records(
    acts_dir: Path, max_chars: int = 1200, overlap: int = 100
) -> tuple[list[dict], dict]:
    records = []
    stats = {
        "files_seen": 0,
        "acts_skipped_repealed": 0,
        "acts_skipped_no_sections": 0,
        "sections_seen": 0,
        "sections_skipped_repealed": 0,
        "sections_skipped_empty": 0,
        "chunks_created": 0,
    }

    for file_path in sorted(acts_dir.glob("act-print-*.json")):
        stats["files_seen"] += 1
        with open(file_path, "r", encoding="utf-8") as f:
            act_obj = json.load(f)

        if act_obj.get("csv_metadata", {}).get("is_repealed") is True:
            stats["acts_skipped_repealed"] += 1
            continue

        sections = act_obj.get("sections") or []
        if not sections:
            stats["acts_skipped_no_sections"] += 1
            continue

        for section in sections:
            stats["sections_seen"] += 1
            raw_content = (section or {}).get("section_content", "")
            if VOID_SECTION_RE.search(raw_content or ""):
                stats["sections_skipped_repealed"] += 1
                continue

            cleaned = clean_section_content(raw_content)
            if not cleaned:
                stats["sections_skipped_empty"] += 1
                continue

            section_index = extract_section_index(cleaned)

            chunks = chunk_section_content(
                cleaned, max_chars=max_chars, overlap=overlap
            )
            for chunk_part, chunk_text in enumerate(chunks, start=1):
                payload = {
                    "act_title": act_obj.get("act_title"),
                    "act_no": act_obj.get("act_no"),
                    "act_year": (
                        int(act_obj["act_year"])
                        if str(act_obj.get("act_year", "")).isdigit()
                        else None
                    ),
                    "section_index": section_index,
                    "chunk_part": chunk_part,
                    "language": act_obj.get("language"),
                    "govt_system": act_obj.get("government_context", {}).get(
                        "govt_system"
                    ),
                    "source_url": act_obj.get("source_url"),
                    "section_content_clean": chunk_text,
                }
                records.append(
                    {
                        "embedding_text": build_embedding_text(
                            act_title=act_obj.get("act_title", "Unknown Act"),
                            section_index=section_index,
                            chunk_part=chunk_part,
                            chunk_text=chunk_text,
                        ),
                        "payload": payload,
                    }
                )

    stats["chunks_created"] = len(records)
    return records, stats

In [ ]:
records, stats = collect_all_records(
    ACTS_DIR, max_chars=MAX_CHARS_PER_CHUNK, overlap=CHUNK_OVERLAP
)

stats

In [ ]:
records[-1]

In [7]:
def get_bedrock_client(runtime: bool = True) -> boto3.client:
    config = Config(
        retries={
            "mode": "adaptive",
        }
    )

    service = "bedrock-runtime" if runtime else "bedrock"

    return boto3.client(
        service_name=service,
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
        region_name=os.getenv("AWS_DEFAULT_REGION"),
        config=config,
    )

In [8]:
bedrock_client = get_bedrock_client()

body = json.dumps(
    {
        "input_type": "search_document",
        "embedding_types": ["float"],
        "texts": ["খনি ও খনিজ সম্পদ (নিয়ন্ত্রণ ও উন্নয়ন) আইন, ১৯৯২", "Mines and Mineral Resources (Control and Development) Act, 1992"],
    }
)

accept = "*/*"
content_type = "application/json"

response = bedrock_client.invoke_model(
    modelId=EMBEDDING_MODEL, body=body, accept=accept, contentType=content_type
)

In [9]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [10]:
response_body = json.loads(response.get("body").read())

embeddings = response_body.get("embeddings")

In [27]:
def embed_texts_bedrock(
    texts: list[str],
    *,
    model_id: str,
    bedrock_client,
    max_input_chars: int = 2048,
) -> list[list[float]]:
    if not texts:
        return []

    # Bedrock embedding APIs enforce a per-text size limit (2048 chars for this model).
    sanitized_texts = [
        (text if len(text) <= max_input_chars else text[:max_input_chars])
        for text in texts
    ]

    body = {
        "input_type": "search_document",
        "embedding_types": ["float"],
        "texts": sanitized_texts,
    }

    response = bedrock_client.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        accept="application/json",
        contentType="application/json",
    )
    result = json.loads(response["body"].read())

    embeddings = result.get("embeddings")
    if isinstance(embeddings, dict) and "float" in embeddings:
        return embeddings["float"]
    if isinstance(embeddings, list):
        return embeddings
    if "embedding" in result:
        return [result["embedding"]]

    raise ValueError(f"Embedding missing in Bedrock response: {result}")

In [34]:
texts_to_embed = ["খনি ও খনিজ সম্পদ (নিয়ন্ত্রণ ও উন্নয়ন) আইন, ১৯৯২", "Khoni o Khonij Shompod (Niyontron o Unnayan) Ain, 1992", "Mines and Mineral Resources (Control and Development) Act, 1992"]
query_to_embed = "Mines and Mineral Resources (Control and Development) Act, 1992"

In [35]:
texts_embedding = embed_texts_bedrock(
    texts=texts_to_embed,
    model_id=EMBEDDING_MODEL,
    bedrock_client=bedrock_client,
)
query_embedding = embed_texts_bedrock(
    texts=[query_to_embed],
    model_id=EMBEDDING_MODEL,
    bedrock_client=bedrock_client,
)

In [36]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a).flatten()
    b = np.array(b).flatten()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

scores = []

for i, doc_emb in enumerate(texts_embedding):
    score = cosine_similarity(query_embedding, doc_emb)
    scores.append((texts_to_embed[i], score))

scores.sort(key=lambda x: x[1], reverse=True)

for doc, score in scores:
    print(f"{doc} -> {score:.4f}")

Mines and Mineral Resources (Control and Development) Act, 1992 -> 1.0000
খনি ও খনিজ সম্পদ (নিয়ন্ত্রণ ও উন্নয়ন) আইন, ১৯৯২ -> 0.7899
Khoni o Khonij Shompod (Niyontron o Unnayan) Ain, 1992 -> 0.5506


In [ ]:
def ensure_qdrant_collection(
    client: QdrantClient, collection_name: str, vector_size: int
) -> None:
    if collection_name not in {c.name for c in client.get_collections().collections}:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=vector_size, distance=models.Distance.COSINE
            ),
        )
    client.create_payload_index(
        collection_name=collection_name,
        field_name="act_year",
        field_schema=models.PayloadSchemaType.INTEGER,
        wait=True,
    )
    client.create_payload_index(
        collection_name=collection_name,
        field_name="language",
        field_schema=models.PayloadSchemaType.KEYWORD,
        wait=True,
    )

In [ ]:
def stable_point_id(source_url: str, section_idx: str | int, chunk_idx: int) -> str:
    return hashlib.sha1(
        f"{source_url}|{section_idx}|{chunk_idx}".encode("utf-8")
    ).hexdigest()

In [ ]:
def ingest_acts_to_qdrant(
    *,
    client: QdrantClient,
    bedrock_client,
    acts_dir: Path,
    collection_name: str,
    model_id: str,
    max_chars: int,
    overlap: int,
    batch_size: int,
    max_records: int | None = None,
) -> dict:
    print("[ingest] Collecting records...")
    records, prep_stats = collect_all_records(
        acts_dir, max_chars=max_chars, overlap=overlap
    )
    print(
        f"[ingest] Files scanned: {prep_stats['files_seen']}, "
        f"chunks prepared: {len(records)}"
    )

    if max_records is not None:
        print(f"[ingest] Applying max_records={max_records}")
        records = records[:max_records]

    stats = {
        "documents_scanned": prep_stats["files_seen"],
        "records_after_filtering": len(records),
        "acts_skipped_repealed": prep_stats["acts_skipped_repealed"],
        "acts_skipped_no_sections": prep_stats["acts_skipped_no_sections"],
        "sections_skipped_repealed": prep_stats["sections_skipped_repealed"],
        "sections_skipped_empty": prep_stats["sections_skipped_empty"],
    }

    if not records:
        print("[ingest] No records to ingest.")
        return {**stats, "points_indexed": 0, "vector_size": None}

    points_indexed = 0
    vector_size = None
    collection_ready = False
    total_batches = (len(records) + batch_size - 1) // batch_size

    print(
        f"[ingest] Starting ingestion: {len(records)} records in {total_batches} batches "
        f"(batch_size={batch_size})"
    )

    for batch_num, start in enumerate(range(0, len(records), batch_size), start=1):
        batch = records[start : start + batch_size]
        print(
            f"[ingest] Batch {batch_num}/{total_batches}: embedding {len(batch)} records..."
        )

        vectors = embed_texts_bedrock(
            [r["embedding_text"] for r in batch],
            model_id=model_id,
            bedrock_client=bedrock_client,
        )

        if len(vectors) != len(batch):
            raise ValueError(
                f"Embedding count mismatch: got {len(vectors)} vectors for {len(batch)} records"
            )

        if not collection_ready:
            vector_size = len(vectors[0])
            print(
                f"[ingest] Ensuring collection '{collection_name}' (vector_size={vector_size})..."
            )
            ensure_qdrant_collection(client, collection_name, vector_size)
            collection_ready = True

        points = []
        for rec, vec in zip(batch, vectors):
            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vec,
                    payload=rec["payload"],
                )
            )

        print(
            f"[ingest] Batch {batch_num}/{total_batches}: writing {len(points)} points to Qdrant..."
        )
        client.upsert(collection_name=collection_name, points=points, wait=True)
        points_indexed += len(points)
        print(f"[ingest] Progress: {points_indexed}/{len(records)} points indexed")

    print("[ingest] Completed.")
    return {**stats, "points_indexed": points_indexed, "vector_size": vector_size}

In [ ]:
qdrant_client = QdrantClient(url=QDRANT_URL, timeout=120)
bedrock_client = get_bedrock_client()

max_records_env = os.getenv("INGEST_MAX_RECORDS", "").strip()
max_records = (
    int(max_records_env)
    if max_records_env.isdigit() and int(max_records_env) > 0
    else None
)

ingestion_summary = ingest_acts_to_qdrant(
    client=qdrant_client,
    bedrock_client=bedrock_client,
    acts_dir=ACTS_DIR,
    collection_name=COLLECTION_NAME,
    model_id=EMBEDDING_MODEL,
    max_chars=MAX_CHARS_PER_CHUNK,
    overlap=CHUNK_OVERLAP,
    batch_size=UPSERT_BATCH_SIZE,
    max_records=max_records,
)

ingestion_summary

In [ ]:
qdrant_client.get_collection(COLLECTION_NAME).model_dump()